# Capítulo 3 — Notebook Maestro de Tablas Econométricas (ZMVM 2015–2019)

Reproduce en orden exacto las Tablas 3.1, 3.2, 3.3 y 3.4 (validación física horaria AERMOD `POSTFILE` por turno escolar).

In [1]:
# === CELDA 1: CARGA DEL PANEL ZMVM (SIN TOLUCA) Y ARMONIZACIÓN DE TURNOS ===
import os, warnings, datetime
import numpy as np
import pandas as pd
import pyfixest as pf

warnings.filterwarnings("ignore", category=UserWarning)

BASE         = "/Users/gabrielevangelista/thesis-cdmx-air"
AERMAP_OUT   = f"{BASE}/02-data/processed/aermap_out"
AERMET_OUT   = f"{BASE}/02-data/processed/aermet_out"
RUN_DIR_EMIS = f"{BASE}/03-analysis/run_aermod_emisfact"
RUN_DIR_BASE = f"{BASE}/03-analysis/run_aermod"
POST_DIR     = f"{BASE}/03-analysis/run_aermod_postfile_95d"

def get_stars(p):
    return "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.10 else ""))

def leer_plt_aermod(filepath):
    records = []
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("*"):
                continue
            p = s.split()
            if len(p) >= 5:
                try:
                    records.append({
                        "X_UTM": round(float(p[0]), 2),
                        "Y_UTM": round(float(p[1]), 2),
                        "CONC":  float(p[2]),
                        "ZELEV": float(p[3]),
                        "ZHILL": float(p[4]),
                    })
                except ValueError:
                    continue
    return pd.DataFrame(records)

# 1. Salidas anuales AERMOD con EMISFACT + corrida plana 24/7
anios_panel = [2015, 2017, 2019]
grupos_plt = {
    "FUNDICION_METAL": "fund",
    "HORNO_VIDRIO":    "vidr",
    "QUIMICA":         "quim",
    "TURBINAS_CFE":    "turb",
    "CALDERAS":        "cald",
}

lista_aermod = []
for yr in anios_panel:
    df_yr = leer_plt_aermod(f"{RUN_DIR_EMIS}/plot_emi_annual_all_{yr}.plt").rename(columns={"CONC": "PM25_ALL"})
    df_yr["anio"] = yr
    for nombre_col, sufijo in grupos_plt.items():
        df_g = leer_plt_aermod(f"{RUN_DIR_EMIS}/plot_emi_annual_{sufijo}_{yr}.plt")[["X_UTM", "Y_UTM", "CONC"]].rename(columns={"CONC": f"PM25_{nombre_col}"})
        df_yr = df_yr.merge(df_g, on=["X_UTM", "Y_UTM"], how="left")
    f_base = f"{RUN_DIR_BASE}/plot_annual_all_{yr}.plt"
    if os.path.exists(f_base):
        df_b = leer_plt_aermod(f_base)[["X_UTM", "Y_UTM", "CONC"]].rename(columns={"CONC": "PM25_BASELINE_24_7"})
        df_yr = df_yr.merge(df_b, on=["X_UTM", "Y_UTM"], how="left")
    lista_aermod.append(df_yr)
df_aermod = pd.concat(lista_aermod, ignore_index=True)

# 2. Meteorología anual AERMET (.sfc)
met_anual = []
for yr in anios_panel:
    rows_sfc = []
    with open(f"{AERMET_OUT}/aer_{yr}.sfc", "r") as f:
        for line in f.readlines()[1:]:
            p = line.split()
            if len(p) >= 22:
                rows_sfc.append({
                    "mo": int(p[1]), "dy": int(p[2]), "hr": int(p[4]),
                    "zic": float(p[9]), "ws": float(p[15]),
                    "temp_k": float(p[18]), "pamt": float(p[21]),
                })
    df_sfc = pd.DataFrame(rows_sfc)
    df_sfc["ws_clean"] = df_sfc["ws"].replace([99.0, 999.0, -9.0], np.nan)
    df_sfc.loc[df_sfc["ws_clean"] > 50, "ws_clean"] = np.nan
    df_sfc["temp_c"] = df_sfc["temp_k"] - 273.15
    df_sfc.loc[(df_sfc["temp_k"] > 330) | (df_sfc["temp_k"] < 240), "temp_c"] = np.nan
    df_sfc["pamt_clean"] = df_sfc["pamt"].where((df_sfc["pamt"] >= 0) & (df_sfc["pamt"] < 200), 0.0)
    daily_t = df_sfc.groupby(["mo", "dy"])["temp_c"].agg(["max", "min"])
    met_anual.append({
        "anio": yr,
        "ws_mean": df_sfc["ws_clean"].mean(),
        "tmax_mean": daily_t["max"].mean(),
        "tmin_mean": daily_t["min"].mean(),
        "precip_sum": df_sfc["pamt_clean"].sum(),
    })
df_met = pd.DataFrame(met_anual)

# 3. Panel PLANEA filtrado a Cuenca de México (X_UTM >= 460 km)
df_planea = pd.read_csv(f"{AERMAP_OUT}/planea_enlazado_receptores_2015_2019.csv")
df_planea["X_UTM"] = df_planea["X_UTM"].round(2)
df_planea["Y_UTM"] = df_planea["Y_UTM"].round(2)
df_planea = df_planea[df_planea["X_UTM"] >= 460000].copy()

mat_raw = 1*df_planea["mat_pct_n1"] + 2*df_planea["mat_pct_n2"] + 3*df_planea["mat_pct_n3"] + 4*df_planea["mat_pct_n4"]
lyc_raw = 1*df_planea["lyc_pct_n1"] + 2*df_planea["lyc_pct_n2"] + 3*df_planea["lyc_pct_n3"] + 4*df_planea["lyc_pct_n4"]
df_planea["mat_std_500"] = 500.0 + 100.0 * (mat_raw - mat_raw.mean()) / mat_raw.std()
df_planea["lyc_std_500"] = 500.0 + 100.0 * (lyc_raw - lyc_raw.mean()) / lyc_raw.std()
df_planea["alta_marg"]   = df_planea["marginacion"].astype(str).str.upper().str.contains("ALTA|MUY ALTA", na=False).astype(float)
df_planea["mun_anio"]    = df_planea["municipio"].astype(str) + "_" + df_planea["anio"].astype(str)

df_reg = df_planea.merge(df_aermod, on=["X_UTM", "Y_UTM", "anio"], how="inner").merge(df_met, on="anio", how="left")

# 4. Armonización de turnos e identificador de unidad escolar
MAPA_TURNO = {
    "1": "MATUTINO", "2": "VESPERTINO", "3": "NOCTURNO", "4": "DISCONTINUO", "5": "TIEMPO COMPLETO",
    "AMPLIADO": "TIEMPO COMPLETO", "COMPLETO": "TIEMPO COMPLETO",
    "JORNADA AMPLIAD": "TIEMPO COMPLETO", "JORNADA AMPLIADA": "TIEMPO COMPLETO",
    "CONTINUO": "TIEMPO COMPLETO", "CONTINUO (TIEMPO COMPLETO)": "TIEMPO COMPLETO",
}
df_reg["turno_norm"] = df_reg["turno"].astype(str).str.upper().str.strip().replace(MAPA_TURNO)
filas_max = df_reg.groupby(["cct", "anio"]).size().groupby("cct").max()
cct_una_fila = set(filas_max[filas_max == 1].index)
df_reg["cct_una_fila"] = df_reg["cct"].isin(cct_una_fila)

def turno_modal(s):
    s_val = s[s != "DISCONTINUO"]
    return (s_val if len(s_val) else s).mode().iloc[0]

turno_cct = df_reg[df_reg["cct_una_fila"]].groupby("cct")["turno_norm"].agg(turno_modal)
df_reg["turno"] = np.where(df_reg["cct_una_fila"], df_reg["cct"].map(turno_cct), df_reg["turno_norm"])
df_reg["id_cct_clean"] = np.where(df_reg["cct_una_fila"], df_reg["cct"].astype(str), df_reg["cct"].astype(str) + "_" + df_reg["turno"].astype(str))

# 5. Controles microclimáticos y transformaciones
z_std  = (df_reg["ZELEV"] - df_reg["ZELEV"].mean()) / df_reg["ZELEV"].std()
zh_std = (df_reg["ZHILL"] - df_reg["ZHILL"].mean()) / df_reg["ZHILL"].std()
df_reg["W_temp_z"]  = df_reg["tmax_mean"] * z_std
df_reg["W_temp_z2"] = (df_reg["tmax_mean"] ** 2) * (z_std ** 2)
df_reg["W_wind_zh"] = df_reg["ws_mean"] * zh_std
df_reg["W_prec_z"]  = (df_reg["precip_sum"] / 100.0) * z_std

df_reg["PM25_x_AltaMarg"] = df_reg["PM25_ALL"] * df_reg["alta_marg"]
df_reg["PM25_x_NoMarg"]   = df_reg["PM25_ALL"] * (1.0 - df_reg["alta_marg"])
df_reg["BASE_x_AltaMarg"] = df_reg["PM25_BASELINE_24_7"] * df_reg["alta_marg"]
df_reg["arcsinh_PM25"]    = np.arcsinh(df_reg["PM25_ALL"] * 100.0)
df_reg["log_PM25"]        = np.log(df_reg["PM25_ALL"] + 0.001)

controles_W = "W_temp_z + W_temp_z2 + W_wind_zh + W_prec_z"
FE = "id_cct_clean + anio"
VC = {"CRV1": "RECID"}

df_pub = df_reg[~df_reg["tipo_escuela"].astype(str).str.upper().str.contains("PRIVAD|PARTICULAR", na=False)].copy()

print(f"[OK] Panel ZMVM listo: N_todas = {len(df_reg):,} | N_publicas = {len(df_pub):,} | CCT = {df_reg['cct'].nunique():,} | RECID = {df_reg['RECID'].nunique():,}")

[OK] Panel ZMVM listo: N_todas = 9,459 | N_publicas = 7,161 | CCT = 2,964 | RECID = 2,392


In [2]:
# === CELDA 2: TABLA 3.1 — ESPECIFICACIÓN CENTRAL (MCO Y TWFE) ===
specs_31 = [
    ("(1) MCO",            "PM25_ALL | anio"),
    ("(2) TWFE base",      "PM25_ALL | id_cct_clean + anio"),
    ("(3) TWFE + met",     f"PM25_ALL + {controles_W} | id_cct_clean + anio"),
    ("(4) TWFE + mun*yr",  f"PM25_ALL + {controles_W} | id_cct_clean + mun_anio"),
]

print("=== TABLA 3.1: ESPECIFICACIÓN CENTRAL (MCO Y TWFE) ===")
for panel, y_std, y_n1 in [("PANEL A: MATEMÁTICAS", "mat_std_500", "mat_pct_n1"),
                           ("PANEL B: LENGUAJE",    "lyc_std_500", "lyc_pct_n1")]:
    print(f"\n{panel}")
    for lbl, rhs in specs_31:
        m = pf.feols(f"{y_std} ~ {rhs}", data=df_reg, vcov=VC)
        r2w = getattr(m, "_r2_within", None)
        r2s = f"{r2w:.4f}" if r2w is not None else "  --  "
        b, se, p = m.coef()["PM25_ALL"], m.se()["PM25_ALL"], m.pvalue()["PM25_ALL"]
        print(f"  {lbl:18s} | Beta = {b:8.3f}{get_stars(p):3s} ({se:6.3f}, p={p:.4f}) | R2w={r2s} | N={m._N:,}")
    m5 = pf.feols(f"{y_n1} ~ PM25_ALL + {controles_W} | {FE}", data=df_reg, vcov=VC)
    b, se, p = m5.coef()["PM25_ALL"], m5.se()["PM25_ALL"], m5.pvalue()["PM25_ALL"]
    print(f"  (5) Nivel I (%)    | Beta = {b:8.3f}{get_stars(p):3s} ({se:6.3f}, p={p:.4f}) | R2w={m5._r2_within:.4f} | N={m5._N:,}")

=== TABLA 3.1: ESPECIFICACIÓN CENTRAL (MCO Y TWFE) ===

PANEL A: MATEMÁTICAS
  (1) MCO            | Beta =   33.209    (28.577, p=0.2453) | R2w=0.0008 | N=9,459
  (2) TWFE base      | Beta =    5.983    ( 8.050, p=0.4574) | R2w=0.0000 | N=9,439
  (3) TWFE + met     | Beta =    5.603    ( 8.080, p=0.4881) | R2w=0.0003 | N=9,439
  (4) TWFE + mun*yr  | Beta =    4.605    ( 9.465, p=0.6266) | R2w=0.0020 | N=9,433
  (5) Nivel I (%)    | Beta =   -1.918    ( 2.041, p=0.3476) | R2w=0.0005 | N=9,439

PANEL B: LENGUAJE
  (1) MCO            | Beta =   66.847    (43.455, p=0.1241) | R2w=0.0030 | N=9,457
  (2) TWFE base      | Beta =    7.554    ( 9.353, p=0.4194) | R2w=0.0001 | N=9,437
  (3) TWFE + met     | Beta =    6.695    ( 9.417, p=0.4772) | R2w=0.0014 | N=9,437
  (4) TWFE + mun*yr  | Beta =    2.237    (12.909, p=0.8625) | R2w=0.0025 | N=9,431
  (5) Nivel I (%)    | Beta =   -2.113    ( 2.320, p=0.3624) | R2w=0.0011 | N=9,437


In [3]:
# === CELDA 3: TABLA 3.2 — HETEROGENEIDAD POR RAMA AP-42 Y POR MARGINACIÓN ===
ramas = ["PM25_FUNDICION_METAL", "PM25_HORNO_VIDRIO", "PM25_QUIMICA", "PM25_TURBINAS_CFE", "PM25_CALDERAS"]
ramas_str = " + ".join(ramas)

print("=== TABLA 3.2: HETEROGENEIDAD POR RAMAS AP-42 Y MARGINACIÓN (6 COLUMNAS) ===")
for asig, y in [("MATEMÁTICAS (Cols 1-3)", "mat_std_500"), ("LENGUAJE (Cols 4-6)", "lyc_std_500")]:
    print(f"\n--- {asig} ---")
    m_ram = pf.feols(f"{y} ~ {ramas_str} + {controles_W} | {FE}", data=df_reg, vcov=VC)
    print(f"  [Col Ramas | N={m_ram._N:,}, R2w={m_ram._r2_within:.4f}]")
    for r in ramas:
        b, se, p = m_ram.coef()[r], m_ram.se()[r], m_ram.pvalue()[r]
        print(f"    {r:22s}: {b:9.3f}{get_stars(p):3s} ({se:7.3f}, p={p:.4f})")

    for sub_lbl, d in [("Todas", df_reg), ("Públicas", df_pub)]:
        m_int = pf.feols(f"{y} ~ PM25_ALL + PM25_x_AltaMarg + {controles_W} | {FE}", data=d, vcov=VC)
        m_tot = pf.feols(f"{y} ~ PM25_x_NoMarg + PM25_x_AltaMarg + {controles_W} | {FE}", data=d, vcov=VC)
        print(f"  [Col Marg {sub_lbl:8s} | N={m_int._N:,}, R2w={m_int._r2_within:.4f}]")
        for lbl_v, mod, v in [
            ("PM25_ALL (base)",           m_int, "PM25_ALL"),
            ("PM25 x AltaMarg (interac)", m_int, "PM25_x_AltaMarg"),
            ("Efecto TOTAL AltaMarg",     m_tot, "PM25_x_AltaMarg"),
        ]:
            b, se, p = mod.coef()[v], mod.se()[v], mod.pvalue()[v]
            print(f"    {lbl_v:26s}: {b:9.3f}{get_stars(p):3s} ({se:7.3f}, p={p:.4f})")

=== TABLA 3.2: HETEROGENEIDAD POR RAMAS AP-42 Y MARGINACIÓN (6 COLUMNAS) ===

--- MATEMÁTICAS (Cols 1-3) ---
  [Col Ramas | N=9,439, R2w=0.0008]
    PM25_FUNDICION_METAL  :     6.664    (  7.881, p=0.3979)
    PM25_HORNO_VIDRIO     :    -2.096    ( 91.552, p=0.9817)
    PM25_QUIMICA          :  -626.760*   (343.258, p=0.0680)
    PM25_TURBINAS_CFE     :   121.023    (237.504, p=0.6104)
    PM25_CALDERAS         :     4.860    ( 94.119, p=0.9588)
  [Col Marg Todas    | N=9,439, R2w=0.0006]
    PM25_ALL (base)           :     8.100    (  8.147, p=0.3202)
    PM25 x AltaMarg (interac) :   -78.095    ( 52.500, p=0.1370)
    Efecto TOTAL AltaMarg     :   -69.995    ( 52.143, p=0.1796)
  [Col Marg Públicas | N=7,138, R2w=0.0004]
    PM25_ALL (base)           :     2.947    (  5.170, p=0.5688)
    PM25 x AltaMarg (interac) :   -56.514    ( 51.794, p=0.2754)
    Efecto TOTAL AltaMarg     :   -53.568    ( 51.655, p=0.2999)

--- LENGUAJE (Cols 4-6) ---
  [Col Ramas | N=9,437, R2w=0.0020]
    PM2

In [4]:
# === CELDA 4: TABLA 3.3 — ROBUSTEZ DE LA INTERACCIÓN DE MARGINACIÓN (SECUNDARIAS PÚBLICAS) ===
print("=== TABLA 3.3 (PANEL A): ROBUSTEZ INTERACCIÓN MARGINACIÓN (AERMOD ANUAL) ===")
for asig, y in [("Lenguaje", "lyc_std_500"), ("Matemáticas", "mat_std_500")]:
    print(f"\n{asig}:")
    specs_rob = [
        ("(1) Central (RECID)",        f"{y} ~ PM25_ALL + PM25_x_AltaMarg + {controles_W} | {FE}",                   VC,                      "PM25_x_AltaMarg"),
        ("(2) Cluster municipio",      f"{y} ~ PM25_ALL + PM25_x_AltaMarg + {controles_W} | {FE}",                   {"CRV1": "municipio"},   "PM25_x_AltaMarg"),
        ("(3) EF municipio x año",     f"{y} ~ PM25_ALL + PM25_x_AltaMarg + {controles_W} | id_cct_clean + mun_anio", VC,                      "PM25_x_AltaMarg"),
        ("(4) Sin controles W",        f"{y} ~ PM25_ALL + PM25_x_AltaMarg | {FE}",                                   VC,                      "PM25_x_AltaMarg"),
        ("(5) Emisión constante 24/7", f"{y} ~ PM25_BASELINE_24_7 + BASE_x_AltaMarg + {controles_W} | {FE}",         VC,                      "BASE_x_AltaMarg"),
    ]
    for lbl, fml, vcov_s, v in specs_rob:
        m = pf.feols(fml, data=df_pub, vcov=vcov_s)
        b, se, p = m.coef()[v], m.se()[v], m.pvalue()[v]
        print(f"  {lbl:28s} | {b:8.3f}{get_stars(p):3s} ({se:6.3f}, p={p:.4f}) | N={m._N:,}")

print("\n=== PRUEBAS ADICIONALES EFECTO PROMEDIO (TODAS Y PÚBLICAS) ===")
for asig, y in [("Lenguaje", "lyc_std_500"), ("Matemáticas", "mat_std_500")]:
    for lbl, d, v in [
        ("log_PM25 (todas)",     df_reg, "log_PM25"),
        ("arcsinh_PM25 (todas)", df_reg, "arcsinh_PM25"),
        ("24/7 plana (todas)",   df_reg, "PM25_BASELINE_24_7"),
        ("PM25_ALL (públicas)",  df_pub, "PM25_ALL"),
    ]:
        m = pf.feols(f"{y} ~ {v} + {controles_W} | {FE}", data=d, vcov=VC)
        print(f"  {asig:11s} | {lbl:21s} | {m.coef()[v]:8.3f} ({m.se()[v]:6.3f}, p={m.pvalue()[v]:.4f})")

=== TABLA 3.3 (PANEL A): ROBUSTEZ INTERACCIÓN MARGINACIÓN (AERMOD ANUAL) ===

Lenguaje:
  (1) Central (RECID)          |  -98.299**  (43.271, p=0.0232) | N=7,138
  (2) Cluster municipio        |  -98.299*** (32.783, p=0.0035) | N=7,138
  (3) EF municipio x año       |  -88.888**  (44.721, p=0.0470) | N=7,129
  (4) Sin controles W          |  -98.620**  (43.028, p=0.0220) | N=7,138
  (5) Emisión constante 24/7   | -107.208**  (46.641, p=0.0217) | N=7,138

Matemáticas:
  (1) Central (RECID)          |  -56.514    (51.794, p=0.2754) | N=7,138
  (2) Cluster municipio        |  -56.514    (43.936, p=0.2015) | N=7,138
  (3) EF municipio x año       |  -58.274    (57.205, p=0.3085) | N=7,129
  (4) Sin controles W          |  -54.289    (51.274, p=0.2898) | N=7,138
  (5) Emisión constante 24/7   |  -59.699    (55.027, p=0.2781) | N=7,138

=== PRUEBAS ADICIONALES EFECTO PROMEDIO (TODAS Y PÚBLICAS) ===
  Lenguaje    | log_PM25 (todas)      |    4.146 ( 2.836, p=0.1439)
  Lenguaje    | arcsinh_PM

In [5]:
# === CELDA 5: CONSTRUCCIÓN DEL INSTRUMENTO EXÓGENO DE VIENTO (30D, 15D, POST30, ENE-MAY) ===
fechas_planea_dt = {
    2015: datetime.date(2015, 6, 17),
    2017: datetime.date(2017, 6, 14),
    2019: datetime.date(2019, 6, 11),
}

df_src = pd.read_csv(f"{AERMAP_OUT}/catalogo_285_fuentes_aermod_ap42.csv")
Xs, Ys, Q_2015 = df_src["X_UTM"].values, df_src["Y_UTM"].values, df_src["Q_GS_2015"].values

rec_unicos = df_reg[["RECID", "X_UTM", "Y_UTM"]].drop_duplicates().reset_index(drop=True)
dx_km = (rec_unicos["X_UTM"].values[:, None] - Xs[None, :]) / 1000.0
dy_km = (rec_unicos["Y_UTM"].values[:, None] - Ys[None, :]) / 1000.0
dist_km = np.maximum(np.sqrt(dx_km**2 + dy_km**2), 0.25)
ux, uy = dx_km / dist_km, dy_km / dist_km
aten_dist = 1.0 / (dist_km ** 1.5)

f_emisfact_24 = np.array([
    0.65, 0.62, 0.60, 0.60, 0.63, 0.72,
    0.88, 1.05, 1.18, 1.25, 1.28, 1.30,
    1.28, 1.25, 1.26, 1.28, 1.26, 1.22,
    1.15, 1.05, 0.92, 0.82, 0.75, 0.68,
])

VENTANAS = {"30D": (0, 30), "15D": (0, 15), "POST30": (-30, -1)}
TURNOS_IV = {"MAT": (7, 13), "VESP": (13, 19)}

def leer_horas_sfc(yr):
    f_exam = fechas_planea_dt[yr]
    filas = []
    with open(f"{AERMET_OUT}/aer_{yr}.sfc", "r") as f:
        for line in f.readlines()[1:]:
            p = line.split()
            if len(p) < 22:
                continue
            mo, dy, hr = int(p[1]), int(p[2]), int(p[4])
            zic, zim = float(p[9]), float(p[10])
            ws, wd = float(p[15]), float(p[16])
            if not (0 < ws < 50 and 0 <= wd <= 360):
                continue
            try:
                fecha = datetime.date(yr, mo, dy)
            except ValueError:
                continue
            z_mix = max([v for v in [zic, zim, 150.0] if 0 < v < 5000])
            trap = f_emisfact_24[(hr - 1) % 24] / (max(ws, 0.5) * np.sqrt(z_mix / 500.0))
            filas.append((mo, (f_exam - fecha).days, fecha.weekday(), hr, wd, trap))
    return pd.DataFrame(filas, columns=["mo", "dias_antes", "dow", "hr", "wd", "trap"])

def indice_cono(sub, Q):
    out = np.zeros(len(rec_unicos))
    if len(sub) == 0:
        return out
    wd_rad = np.deg2rad(sub["wd"].values)
    wx_u, wy_u = -np.sin(wd_rad), -np.cos(wd_rad)
    sectores = np.floor(sub["wd"].values / 10.0).astype(int) % 36
    trap = sub["trap"].values
    for sec in range(36):
        m = (sectores == sec)
        if not np.any(m):
            continue
        wx_m, wy_m = np.mean(wx_u[m]), np.mean(wy_u[m])
        norm = max(np.sqrt(wx_m**2 + wy_m**2), 1e-6)
        cono = np.maximum(0.0, ux * (wx_m / norm) + uy * (wy_m / norm)) ** 10
        out += np.sum(cono * (aten_dist * Q[None, :]), axis=1) * (np.sum(trap[m]) / len(sub))
    return out

registros_v = []
for yr in fechas_planea_dt:
    Q_yr = df_src[f"Q_GS_{yr}"].values
    df_h = leer_horas_sfc(yr)
    df_h = df_h[(df_h["dow"] < 5) & (df_h["hr"].between(7, 19))]
    res = rec_unicos[["RECID"]].copy()
    res["anio"] = yr
    # Ventanas 30D, 15D y POST30
    for w_nom, (d_ini, d_fin) in VENTANAS.items():
        for t_nom, (h1, h2) in TURNOS_IV.items():
            sub = df_h[df_h["dias_antes"].between(d_ini, d_fin) & df_h["hr"].between(h1, h2)]
            res[f"IVRAW_{w_nom}_{t_nom}"] = indice_cono(sub, Q_2015)
            res[f"EXP_{w_nom}_{t_nom}"]   = indice_cono(sub, Q_yr)
    # Semestre lectivo Ene-May (para Tabla 3.3 Panel B)
    sub_em = df_h[df_h["mo"].between(1, 5)]
    res["EXP_ENEMAY_ALL"] = indice_cono(sub_em, Q_yr)
    for t_nom, (h1, h2) in TURNOS_IV.items():
        res[f"EXP_ENEMAY_{t_nom}"] = indice_cono(sub_em[sub_em["hr"].between(h1, h2)], Q_yr)
    registros_v.append(res)

df_v = pd.concat(registros_v, ignore_index=True)

escala_30 = df_reg["PM25_ALL"].mean() / df_v["EXP_30D_MAT"].mean()
escala_em = df_reg["PM25_ALL"].mean() / df_v["EXP_ENEMAY_ALL"].mean()

for c in [c for c in df_v.columns if c.startswith("IVRAW_")]:
    df_v[c.replace("IVRAW_", "IV_")] = (df_v[c] - df_v[c].mean()) / df_v[c].std()
for w_nom in VENTANAS:
    for t_nom in TURNOS_IV:
        df_v[f"PM25_{w_nom}_{t_nom}"] = df_v[f"EXP_{w_nom}_{t_nom}"] * escala_30
df_v["PM25_LECTIVO"] = df_v["EXP_ENEMAY_ALL"] * escala_em
df_v["PM25_EM_MAT"]  = df_v["EXP_ENEMAY_MAT"] * escala_em
df_v["PM25_EM_VESP"] = df_v["EXP_ENEMAY_VESP"] * escala_em

cols_keep = [c for c in df_v.columns if c.startswith(("IV_", "PM25_"))]
df_reg = df_reg.merge(df_v[["RECID", "anio"] + cols_keep], on=["RECID", "anio"], how="left")

es_vesp = df_reg["turno"].astype(str).str.upper().str.contains("VESP|2").values
for w_nom in VENTANAS:
    df_reg[f"IV_{w_nom}"]   = np.where(es_vesp, df_reg[f"IV_{w_nom}_VESP"], df_reg[f"IV_{w_nom}_MAT"])
    df_reg[f"PM25_{w_nom}"] = np.where(es_vesp, df_reg[f"PM25_{w_nom}_VESP"], df_reg[f"PM25_{w_nom}_MAT"])
    df_reg[f"PM25_{w_nom}_x_Marg"] = df_reg[f"PM25_{w_nom}"] * df_reg["alta_marg"]
df_reg["PM25_EM_TURNO"]        = np.where(es_vesp, df_reg["PM25_EM_VESP"], df_reg["PM25_EM_MAT"])
df_reg["PM25_LECTIVO_x_Marg"]  = df_reg["PM25_LECTIVO"] * df_reg["alta_marg"]
df_reg["PM25_EM_TURNO_x_Marg"] = df_reg["PM25_EM_TURNO"] * df_reg["alta_marg"]

df_pub = df_reg[~df_reg["tipo_escuela"].astype(str).str.upper().str.contains("PRIVAD|PARTICULAR", na=False)].copy()

print("=== TABLA 3.3 (PANEL B): ROBUSTEZ INTERACCIÓN EN SEMESTRE LECTIVO ENE-MAY ===")
for lbl, v_base, v_int in [
    ("(6) Ene-May 07-19h",     "PM25_LECTIVO",  "PM25_LECTIVO_x_Marg"),
    ("(7) Ene-May por turno",  "PM25_EM_TURNO", "PM25_EM_TURNO_x_Marg"),
]:
    m_l = pf.feols(f"lyc_std_500 ~ {v_base} + {v_int} + {controles_W} | {FE}", data=df_pub, vcov=VC)
    m_m = pf.feols(f"mat_std_500 ~ {v_base} + {v_int} + {controles_W} | {FE}", data=df_pub, vcov=VC)
    print(f"  {lbl:22s} | LyC={m_l.coef()[v_int]:8.3f}{get_stars(m_l.pvalue()[v_int]):3s} ({m_l.se()[v_int]:6.3f}, p={m_l.pvalue()[v_int]:.4f}) | Mat={m_m.coef()[v_int]:8.3f} ({m_m.se()[v_int]:6.3f}, p={m_m.pvalue()[v_int]:.4f})")

=== TABLA 3.3 (PANEL B): ROBUSTEZ INTERACCIÓN EN SEMESTRE LECTIVO ENE-MAY ===
  (6) Ene-May 07-19h     | LyC= -49.034**  (23.043, p=0.0335) | Mat= -12.476 (31.710, p=0.6940)
  (7) Ene-May por turno  | LyC= -26.262    (21.915, p=0.2309) | Mat=   1.809 (32.540, p=0.9557)


In [6]:
# === CELDA 6: MATRIZ SIMÉTRICA 2x2 (TWFE FÍSICO vs 2SLS FÍSICO POSTFILE) Y TABLA 3.4 ===
import numpy as np
import pandas as pd
import pyfixest as pf

# 1. Recuperar diagnóstico del IV Anual (justifica por qué el Pilar 2 migra a 30 días)
rec_unicos = df_reg[["RECID", "X_UTM", "Y_UTM"]].drop_duplicates().reset_index(drop=True)
dx_km = (rec_unicos["X_UTM"].values[:, None] - Xs[None, :]) / 1000.0
dy_km = (rec_unicos["Y_UTM"].values[:, None] - Ys[None, :]) / 1000.0
dist_km = np.maximum(np.sqrt(dx_km**2 + dy_km**2), 0.25)
ux, uy = dx_km / dist_km, dy_km / dist_km
peso_f = Q_2015[None, :] / (dist_km ** 1.5)

iv_anual_list = []
for yr in [2015, 2017, 2019]:
    wd_l, ws_l = [], []
    with open(f"{AERMET_OUT}/aer_{yr}.sfc", "r") as f:
        for line in f.readlines()[1:]:
            p = line.split()
            if len(p) >= 22 and 0 < float(p[15]) < 50 and 0 <= float(p[16]) <= 360:
                ws_l.append(float(p[15])); wd_l.append(float(p[16]))
    ws_a, wd_r = np.array(ws_l), np.deg2rad(np.array(wd_l))
    wx_u, wy_u, inv_u = -np.sin(wd_r), -np.cos(wd_r), 1.0 / np.maximum(ws_a, 0.5)
    sec_a = np.floor(np.array(wd_l) / 10.0).astype(int) % 36
    acc = np.zeros(len(rec_unicos))
    for s in range(36):
        m = (sec_a == s)
        if np.any(m):
            wx_m, wy_m = np.mean(wx_u[m]), np.mean(wy_u[m])
            nm = max(np.sqrt(wx_m**2 + wy_m**2), 1e-6)
            acc += np.sum(np.maximum(0.0, ux*(wx_m/nm) + uy*(wy_m/nm))**8 * peso_f, axis=1) * (np.sum(inv_u[m])/len(ws_a))
    iv_anual_list.append(pd.DataFrame({"RECID": rec_unicos["RECID"], "anio": yr, "IV_ANUAL_RAW": acc}))

df_iva = pd.concat(iv_anual_list, ignore_index=True)
df_iva["IV_ANUAL"] = (df_iva["IV_ANUAL_RAW"] - df_iva["IV_ANUAL_RAW"].mean()) / df_iva["IV_ANUAL_RAW"].std()
w_sd_anual = (df_iva["IV_ANUAL"] - df_iva.groupby("RECID")["IV_ANUAL"].transform("mean") - df_iva.groupby("anio")["IV_ANUAL"].transform("mean")).std()

df_pub_a = df_pub.merge(df_iva[["RECID", "anio", "IV_ANUAL"]], on=["RECID", "anio"], how="left")
fs_an = pf.feols(f"PM25_ALL ~ IV_ANUAL + {controles_W} | {FE}", data=df_pub_a, vcov=VC, collin_tol=1e-12)
iv_an = pf.feols(f"lyc_std_500 ~ {controles_W} + [PM25_ALL ~ IV_ANUAL] | {FE}", data=df_pub_a, vcov=VC, collin_tol=1e-12)

print("=== DIAGNÓSTICO DE MOTIVACIÓN PARA EL PILAR 2 (IV ANUAL vs 30 DÍAS) ===")
print(f"  IV Anual (8,760h): DE_within = {w_sd_anual:.5f} | F1 = {fs_an.tstat()['IV_ANUAL']**2:.2f} | 2SLS_LyC = {iv_an.coef()['PM25_ALL']:.2f} (SE = {iv_an.se()['PM25_ALL']:.2f})")

# 2. Procesar salidas físicas horarias POSTFILE (ug/m3)
FECHAS_TS = {2015: pd.Timestamp("2015-06-17"), 2017: pd.Timestamp("2017-06-14"), 2019: pd.Timestamp("2019-06-11")}
HORAS_TURNO = {
    "MATUTINO": set(range(8, 15)), "VESPERTINO": set(range(14, 21)),
    "TIEMPO COMPLETO": set(range(9, 17)), "NOCTURNO": set(range(18, 23)), "DISCONTINUO": set(range(8, 20)),
}

def procesar_postfile(filepath, anio):
    df = pd.read_csv(filepath, sep=r"\s+", skiprows=8, usecols=[0, 1, 2, 8], names=["X_UTM", "Y_UTM", "CONC", "DATE"],
                     dtype={"X_UTM": np.float64, "Y_UTM": np.float64, "CONC": np.float32, "DATE": np.int64})
    df["X_UTM"], df["Y_UTM"] = df["X_UTM"].round(2), df["Y_UTM"].round(2)
    df = df[df["X_UTM"] >= 460000].copy()
    hr, ymd = (df["DATE"] % 100).astype(np.int8), df["DATE"] // 100
    fecha = pd.to_datetime({"year": (ymd // 10000) + 2000, "month": (ymd // 100) % 100, "day": ymd % 100})
    dias_rel = (fecha - FECHAS_TS[anio]).dt.days

    def agg_ventana(mask, sufijo):
        sub = df.loc[mask, ["X_UTM", "Y_UTM", "CONC"]].copy()
        sub["HR"] = hr[mask].values
        res = sub.groupby(["X_UTM", "Y_UTM"], as_index=False)["CONC"].mean().rename(columns={"CONC": f"24h_{sufijo}"})
        for t_lbl, hrs_set in HORAS_TURNO.items():
            gt = sub[sub["HR"].isin(hrs_set)].groupby(["X_UTM", "Y_UTM"], as_index=False)["CONC"].mean()
            res = res.merge(gt.rename(columns={"CONC": f"{t_lbl}_{sufijo}"}), on=["X_UTM", "Y_UTM"], how="left")
        return res

    out = agg_ventana((dias_rel >= -30) & (dias_rel <= -1), "30d_prev")
    out = out.merge(agg_ventana((dias_rel >= -15) & (dias_rel <= -1), "15d_prev"), on=["X_UTM", "Y_UTM"])
    out = out.merge(agg_ventana((dias_rel >= 1)   & (dias_rel <= 30), "30d_post"), on=["X_UTM", "Y_UTM"])
    out["anio"] = anio
    return out

df_real = pd.concat([procesar_postfile(f"{POST_DIR}/post_real_{yr}.plt", yr) for yr in [2015, 2017, 2019]], ignore_index=True)
df_real = df_real.rename(columns={c: f"E_{c}" for c in df_real.columns if c not in ["X_UTM", "Y_UTM", "anio"]})
df_fis = df_pub.merge(df_real, on=["X_UTM", "Y_UTM", "anio"], how="inner")

for vent in ["30d_prev", "15d_prev", "30d_post"]:
    col_dest = f"E_turno_{vent}"
    df_fis[col_dest] = np.nan
    for t_lbl in HORAS_TURNO:
        m = df_fis["turno"] == t_lbl
        df_fis.loc[m, col_dest] = df_fis.loc[m, f"E_{t_lbl}_{vent}"]

sub_todas = df_fis
sub_vuln  = df_fis[df_fis["alta_marg"] == 1]
sub_novul = df_fis[df_fis["alta_marg"] == 0]

# 3. TABLA 3.4 SIMÉTRICA: TWFE FÍSICO vs FORMA REDUCIDA vs 2SLS SOBRE LOS MISMOS DATOS
columnas_34 = [
    ("(1) Todas públicas",        sub_todas, "E_turno_30d_prev", "IV_30D",    "anio"),
    ("(2) Alta Marg.",            sub_vuln,  "E_turno_30d_prev", "IV_30D",    "anio"),
    ("(3) Alta Marg. (Mun*Año)",  sub_vuln,  "E_turno_30d_prev", "IV_30D",    "mun_anio"),
    ("(4) Baja/Media Marg.",      sub_novul, "E_turno_30d_prev", "IV_30D",    "anio"),
    ("(5) Alta Marg. (15d)",      sub_vuln,  "E_turno_15d_prev", "IV_15D",    "anio"),
    ("(6) Placebo Post (30d)",    sub_vuln,  "E_turno_30d_post", "IV_POST30", "anio"),
]

print("\n=== TABLA 3.4 SIMÉTRICA: COMPARACIÓN DIRECTA TWFE FÍSICO vs FORMA REDUCIDA vs 2SLS (ug/m3) ===")
for panel_lbl, y_var in [("PANEL A: LENGUAJE Y COMUNICACIÓN", "lyc_std_500"), ("PANEL B: MATEMÁTICAS", "mat_std_500")]:
    print(f"\n{panel_lbl}")
    for col_lbl, d, e_c, z_c, fe_t in columnas_34:
        tw = pf.feols(f"{y_var} ~ {e_c} + {controles_W} | id_cct_clean + {fe_t}", data=d, vcov=VC)
        rf = pf.feols(f"{y_var} ~ {z_c} + {controles_W} | id_cct_clean + {fe_t}", data=d, vcov=VC)
        iv = pf.feols(f"{y_var} ~ {controles_W} + [{e_c} ~ {z_c}] | id_cct_clean + {fe_t}", data=d, vcov=VC, collin_tol=1e-12)
        fs = pf.feols(f"{e_c} ~ {z_c} + {controles_W} | id_cct_clean + {fe_t}", data=d, vcov=VC, collin_tol=1e-12)
        f1 = fs.tstat()[z_c] ** 2
        bt, set_, pt = tw.coef()[e_c], tw.se()[e_c], tw.pvalue()[e_c]
        br, ser, pr  = rf.coef()[z_c], rf.se()[z_c], rf.pvalue()[z_c]
        bi, sei, pi  = iv.coef()[e_c], iv.se()[e_c], iv.pvalue()[e_c]
        print(f"  {col_lbl:25s} | TWFE={bt:7.2f}{get_stars(pt):3s} ({set_:6.2f}, p={pt:.3f}) | RF={br:7.3f}{get_stars(pr):3s} ({ser:5.3f}, p={pr:.3f}) | 2SLS={bi:8.2f}{get_stars(pi):3s} ({sei:6.2f}, p={pi:.3f}) | F1={f1:5.1f} | N={iv._N:,}")

=== DIAGNÓSTICO DE MOTIVACIÓN PARA EL PILAR 2 (IV ANUAL vs 30 DÍAS) ===
  IV Anual (8,760h): DE_within = 0.05120 | F1 = 2.64 | 2SLS_LyC = -378.44 (SE = 304.32)

=== TABLA 3.4 SIMÉTRICA: COMPARACIÓN DIRECTA TWFE FÍSICO vs FORMA REDUCIDA vs 2SLS (ug/m3) ===

PANEL A: LENGUAJE Y COMUNICACIÓN
  (1) Todas públicas        | TWFE=  -1.03    (  3.93, p=0.794) | RF= -4.870**  (1.928, p=0.012) | 2SLS= -172.20*   ( 89.71, p=0.055) | F1=  8.9 | N=7,138
  (2) Alta Marg.            | TWFE= -41.59    ( 36.38, p=0.254) | RF=-10.700**  (4.823, p=0.027) | 2SLS= -450.38*** (163.98, p=0.006) | F1=  9.2 | N=2,011
  (3) Alta Marg. (Mun*Año)  | TWFE=  10.19    ( 56.37, p=0.857) | RF=-18.435**  (7.819, p=0.019) | 2SLS=-1098.43**  (504.96, p=0.030) | F1= 12.6 | N=1,986
  (4) Baja/Media Marg.      | TWFE=  -0.33    (  3.71, p=0.930) | RF= -3.809*   (2.195, p=0.083) | 2SLS= -133.71    (101.33, p=0.187) | F1=  7.2 | N=5,096
  (5) Alta Marg. (15d)      | TWFE= -46.82    ( 39.75, p=0.240) | RF= -6.931    (4.823, p=